In [2]:
from transformers import pipeline

print("⏳ 모델 다운로드 및 불러오는 중... (첫 실행 시 1~2분 정도 걸립니다)")

# 1. 스마일게이트 kor_unsmile 모델 파이프라인 생성 (CPU로 자동 실행)
classifier = pipeline(
    "text-classification",
    model="smilegate-ai/kor_unsmile",
    top_k=None  # 10개 혐오/욕설 카테고리 점수를 전부 다 확인하기 위함
)

print("✅ 모델 로딩 성공!\n")

# 2. 테스트 문장 넣어보기
test_text = "이 영화 진짜 개재밌다 꼭 봐라"  # 정상 문장 테스트
# test_text = "야 이 개새끼야 꺼져라"      # 일반 욕설 테스트

# 3. 판별 실행
results = classifier(test_text)

# 4. 결과 출력
print(f"입력 문장: '{test_text}'")
print("-" * 50)
for result in results[0]:
    label = result['label']
    score = result['score'] * 100
    # 확률이 50%가 넘는 항목만 강조 출력
    if score > 50:
        print(f"🚨 [감지됨] {label}: {score:.2f}%")
    else:
        print(f"   {label}: {score:.2f}%")

⏳ 모델 다운로드 및 불러오는 중... (첫 실행 시 1~2분 정도 걸립니다)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 모델 로딩 성공!

입력 문장: '이 영화 진짜 개재밌다 꼭 봐라'
--------------------------------------------------
🚨 [감지됨] clean: 88.13%
   악플/욕설: 9.36%
   여성/가족: 1.76%
   인종/국적: 1.58%
   지역: 1.22%
   성소수자: 1.19%
   종교: 1.04%
   남성: 1.03%
   기타 혐오: 0.86%
   연령: 0.77%


In [3]:
test_text = "씳빫아"

results = classifier(test_text)

# 4. 결과 출력
print(f"입력 문장: '{test_text}'")
print("-" * 50)
for result in results[0]:
    label = result['label']
    score = result['score'] * 100
    # 확률이 50%가 넘는 항목만 강조 출력
    if score > 50:
        print(f"🚨 [감지됨] {label}: {score:.2f}%")
    else:
        print(f"   {label}: {score:.2f}%")

입력 문장: '씳빫아'
--------------------------------------------------
🚨 [감지됨] clean: 85.95%
   악플/욕설: 10.77%
   인종/국적: 1.33%
   지역: 1.29%
   여성/가족: 1.22%
   성소수자: 1.13%
   종교: 1.09%
   남성: 0.98%
   연령: 0.80%
   기타 혐오: 0.79%


In [4]:
import re
from transformers import pipeline

# ====================================================================
# [1] 한글 자모 분해 및 겹받침/노이즈 정규화 함수
# ====================================================================
CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

# 우회용으로 자주 쓰이는 겹받침 단순화 매핑
COMPLEX_BATCHIM_MAP = {
    'ㄼ': '',    # 씗 -> 씨
    'ㅀ': '',    # 싷 -> 시
    'ㄶ': '',    # 싾 -> 싸
    'ㄻ': 'ㄹ',  # 빪 -> 빨, 밞 -> 발
    'ㄺ': 'ㄱ',
    'ㄳ': 'ㄱ',
    'ㅄ': 'ㅂ'
}

def decompose_hangul(char):
    """한글 한 글자를 초성, 중성, 종성으로 분해"""
    if '가' <= char <= '힣':
        code = ord(char) - 0xAC00
        cho = code // (21 * 28)
        jung = (code % (21 * 28)) // 28
        jong = code % 28
        return CHOSUNG[cho], JUNGSUNG[jung], JONGSUNG[jong]
    return char, '', ''

def compose_hangul(cho, jung, jong=''):
    """초성, 중성, 종성을 하나의 한글 글자로 결합"""
    if cho in CHOSUNG and jung in JUNGSUNG:
        cho_idx = CHOSUNG.index(cho)
        jung_idx = JUNGSUNG.index(jung)
        jong_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
        return chr(0xAC00 + (cho_idx * 21 + jung_idx) * 28 + jong_idx)
    return cho

def clean_bypassed_text(text):
    """변형 욕설(특수문자, 이상한 겹받침)을 원형으로 복원하는 전처리 엔진"""
    # 1. 글자 사이에 끼워넣은 숫자나 특수문자 제거 (예: 시1발 -> 시발)
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}]', '', text)
    
    # 2. 겹받침 노이즈 정규화 (예: 씗 -> 씨, 빪 -> 빨)
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            cho, jung, jong = decompose_hangul(char)
            # 겹받침이 들어가 있다면 단순화
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_hangul(cho, jung, jong))
        else:
            normalized_chars.append(char)
            
    cleaned_text = "".join(normalized_chars)
    
    # 3. 흔한 쌍자음 변형 원형 복원 (씨빨 -> 씨발)
    cleaned_text = cleaned_text.replace("씨빨", "씨발").replace("시빨", "시발")
    return cleaned_text


# ====================================================================
# [2] 모델 로딩 및 전처리 전/후 비교 테스트
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
print("✅ 로딩 완료!\n")

def check_text(text):
    results = classifier(text)
    for res in results[0]:
        if res['label'] == '악플/욕설':
            return res['score'] * 100
    return 0.0

# ⭐️ 아까 감지 실패했던 '씗빪' 테스트 문장
test_sentence = "씳빪아"

print("=" * 60)
print(f"📌 [테스트 입력 문장]: '{test_sentence}'")
print("=" * 60)

# 1. 전처리 없이 그냥 넣었을 때
score_before = check_text(test_sentence)
print(f"❌ 1. 전처리 전 (기존 모델만 사용):")
print(f"   - 감지된 악플/욕설 확률: {score_before:.2f}%")
if score_before < 50:
    print("   👉 [결과: 뚫림 (감지 실패!)]")

print("-" * 60)

# 2. 우리 전처리 엔진을 거쳐서 넣었을 때
preprocessed_sentence = clean_bypassed_text(test_sentence)
score_after = check_text(preprocessed_sentence)
print(f"⭕ 2. 우리 전처리 엔진 통과 후:")
print(f"   - 복원된 문장: '{preprocessed_sentence}'")
print(f"   - 감지된 악플/욕설 확률: {score_after:.2f}%")
if score_after >= 50:
    print("   👉 🚨 [결과: 완벽 감지 및 검거 성공!]")
print("=" * 60)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

📌 [테스트 입력 문장]: '씳빪아'
❌ 1. 전처리 전 (기존 모델만 사용):
   - 감지된 악플/욕설 확률: 10.77%
   👉 [결과: 뚫림 (감지 실패!)]
------------------------------------------------------------
⭕ 2. 우리 전처리 엔진 통과 후:
   - 복원된 문장: '씨발아'
   - 감지된 악플/욕설 확률: 92.04%
   👉 🚨 [결과: 완벽 감지 및 검거 성공!]


In [11]:
from transformers import pipeline

print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
print("✅ 로딩 완료!\n")

# 🧪 네티즌들이 자주 쓰는 기상천외한 우회 욕설 테스트셋
test_cases = [
    # [1] 자모 쪼개기 유형
    "야 이 ㅅㅣ1발롬아",
    "진짜 ㅂㅕㅇㅅㅣㄴ 같네",
    "ㅈㅗㅈ까고 있네 진짜",
    "ㅅㅣㅂㅏㄹ 저리 가라",
    
    # [2] 특수문자 / 기호 삽입 유형
    "시-발 진짜 짜증나네",
    "시@발 장난하나",
    "병~신아 꺼져라",
    "개!새!끼야 나와라",
    
    # [3] 숫자 치환 유형
    "18롬아 조용히 해",
    "존1나 열받네",
    "alcls tlqkf",
    
    # [4] 정상 문장 (대조군: 오탐하면 안 되는 문장)
    "이번 프로젝트는 새로운 시발점이 되었다",
    "우리 조카 신발 너무 귀엽다"
]

print("=" * 70)
print(f"{'테스트 문장':<25} | {'악플/욕설 확률':<12} | {'판정 결과'}")
print("=" * 70)

hacked_count = 0  # 뚫린(감지 실패한) 욕설 개수
total_swear_count = 10  # 위 10개는 욕설

for text in test_cases:
    results = classifier(text)
    
    # 악플/욕설 점수 추출
    swear_score = 0.0
    for res in results[0]:
        if res['label'] == '악플/욕설':
            swear_score = res['score'] * 100
            break
            
    # 판정 (50% 이상이면 감지 성공)
    if "시발점" in text or "조카 신발" in text:
        # 정상 문장인 경우
        status = "✅ 정상 통과 (정상)" if swear_score < 50 else "⚠️ 오탐 (정상 문장 차단됨)"
    else:
        # 욕설 문장인 경우
        if swear_score >= 50:
            status = "🚨 [감지 성공]"
        else:
            status = "❌ [뚫림! 감지 실패]"
            hacked_count += 1
            
    print(f"{text:<25} | {swear_score:>10.2f}% | {status}")

print("=" * 70)
print(f"📊 [결과 요약]")
print(f"- 우회 욕설 10개 중 뚫린 개수: {hacked_count}개")
print(f"- 기존 모델의 우회 욕설 탐지율: {((total_swear_count - hacked_count) / total_swear_count) * 100:.1f}%")
print("=" * 70)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

테스트 문장                    | 악플/욕설 확률     | 판정 결과
야 이 ㅅㅣ1발롬아                |      20.31% | ❌ [뚫림! 감지 실패]
진짜 ㅂㅕㅇㅅㅣㄴ 같네              |      78.18% | 🚨 [감지 성공]
ㅈㅗㅈ까고 있네 진짜               |      57.97% | 🚨 [감지 성공]
ㅅㅣㅂㅏㄹ 저리 가라               |      20.81% | ❌ [뚫림! 감지 실패]
시-발 진짜 짜증나네               |      91.19% | 🚨 [감지 성공]
시@발 장난하나                  |      90.82% | 🚨 [감지 성공]
병~신아 꺼져라                  |      91.74% | 🚨 [감지 성공]
개!새!끼야 나와라                |      89.64% | 🚨 [감지 성공]
18롬아 조용히 해                |      87.95% | 🚨 [감지 성공]
존1나 열받네                   |      87.15% | 🚨 [감지 성공]
alcls tlqkf               |       5.76% | ❌ [뚫림! 감지 실패]
이번 프로젝트는 새로운 시발점이 되었다     |       8.31% | ✅ 정상 통과 (정상)
우리 조카 신발 너무 귀엽다           |       5.90% | ✅ 정상 통과 (정상)
📊 [결과 요약]
- 우회 욕설 10개 중 뚫린 개수: 3개
- 기존 모델의 우회 욕설 탐지율: 70.0%


In [6]:
import re
from transformers import pipeline

# ====================================================================
# [1] 자모 결합 및 겹받침 정규화 엔진
# ====================================================================
CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

COMPLEX_BATCHIM_MAP = {
    'ㄼ': '',    # 씗 -> 씨
    'ㅀ': '',    # 싷 -> 시
    'ㄶ': '',    # 싾 -> 싸
    'ㄻ': 'ㄹ',  # 빪 -> 빨, 밞 -> 발
    'ㄺ': 'ㄱ',
    'ㄳ': 'ㄱ',
    'ㅄ': 'ㅂ'
}

def compose_syllable(cho, jung, jong=''):
    """초성, 중성, 종성을 하나의 완성된 한글 글자로 결합"""
    c_idx = CHOSUNG.index(cho)
    j_idx = JUNGSUNG.index(jung)
    t_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
    return chr(0xAC00 + (c_idx * 21 + j_idx) * 28 + t_idx)

def combine_jamo(text):
    """쪼개진 자모(ㅅ, ㅣ, ㅂ, ㅏ, ㄹ)를 완성형 글자(시, 발)로 조립하는 오토마타"""
    chars = list(text)
    result = []
    i = 0
    n = len(chars)
    
    while i < n:
        c = chars[i]
        # 초성 + 중성이 연속으로 등장하는지 확인
        if c in CHOSUNG and i + 1 < n and chars[i+1] in JUNGSUNG:
            cho = c
            jung = chars[i+1]
            jong = ""
            
            # 다음 글자가 종성(받침) 후보인지 확인
            if i + 2 < n and chars[i+2] in JONGSUNG and chars[i+2] != "":
                # 만약 그 다다음 글자가 또 모음(중성)이라면, 이 받침은 다음 글자의 초성으로 양보!
                # 예: 'ㅅ','ㅣ','ㅂ','ㅏ','ㄹ' 에서 'ㅂ'은 '시'의 받침이 아니라 '발'의 초성임
                if i + 3 < n and chars[i+3] in JUNGSUNG:
                    jong = ""
                    i += 2
                else:
                    jong = chars[i+2]
                    i += 3
            else:
                i += 2
                
            result.append(compose_syllable(cho, jung, jong))
        else:
            result.append(c)
            i += 1
            
    return "".join(result)

def clean_bypassed_text(text):
    """특수문자 제거 + 자모 결합 + 겹받침 정규화를 한 번에 수행"""
    # 1. 자모 사이에 끼어든 숫자/특수문자 제거 (예: ㅅㅣ1발 -> ㅅㅣ발)
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}\[\]\-]', '', text)
    
    # 2. 쪼개진 자모를 완성형 글자로 결합 (예: ㅅㅣ발 -> 시발, ㅈㅗㅈ -> 좆)
    text = combine_jamo(text)
    
    # 3. 씗빪 같은 이상한 겹받침 분해 후 단순화
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - 0xAC00
            cho = CHOSUNG[code // (21 * 28)]
            jung = JUNGSUNG[(code % (21 * 28)) // 28]
            jong = JONGSUNG[code % 28]
            
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_syllable(cho, jung, jong))
        else:
            normalized_chars.append(char)
            
    cleaned_text = "".join(normalized_chars)
    
    # 4. 발음 변형 마무리
    cleaned_text = cleaned_text.replace("씨빨", "씨발").replace("시빨", "시발")
    return cleaned_text


# ====================================================================
# [2] 아까 뚫렸던 문장들 전처리 전/후 비교 테스트
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
print("✅ 로딩 완료!\n")

def get_swear_score(text):
    results = classifier(text)
    for res in results[0]:
        if res['label'] == '악플/욕설':
            return res['score'] * 100
    return 0.0

# 🧪 아까 뚫리거나 위태로웠던 문장 4인방
test_sentences = [
    "야 이 ㅅㅣ1발롬아",
    "ㅅㅣㅂㅏㄹ 저리 가라",
    "ㅈㅗㅈ까고 있네 진짜",
    "야 이 씗빪아",
    "시...발...새...끼",
    "조 까 구 있 네",
    "개@새@끼",
    "시ㅂㅏㄹ련아",
    "ㅅ1ㅂㅏㄹ",
    "꺼 ㅈㅕ",
    "시베리아 귤껍데기",
    "개색구 야",
    "신발끈 조카신발",
    "수박 씨 발라먹어",
    "tlqkf",
    "shdem",
    "개ㅅH끼",
    "앰창",
    "느금마",
    
]

print("=" * 80)
print(f"{'원래 문장':<20} ➔ {'복원된 문장':<18} | {'전처리 전':<10} ➔ {'전처리 후'}")
print("=" * 80)

for original in test_sentences:
    # 1. 전처리 전 점수
    score_before = get_swear_score(original)
    
    # 2. 자모 결합 엔진 통과
    restored = clean_bypassed_text(original)
    
    # 3. 전처리 후 점수
    score_after = get_swear_score(restored)
    
    status = "🚨 검거 성공!" if score_after >= 50 else "❌ 여전히 실패"
    print(f"{original:<20} ➔ {restored:<18} | {score_before:>8.2f}% ➔ {score_after:>8.2f}% | {status}")

print("=" * 80)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

원래 문장                ➔ 복원된 문장             | 전처리 전      ➔ 전처리 후
야 이 ㅅㅣ1발롬아           ➔ 야 이 시발롬아           |    20.31% ➔    91.76% | 🚨 검거 성공!
ㅅㅣㅂㅏㄹ 저리 가라          ➔ 시발 저리 가라           |    20.81% ➔    90.96% | 🚨 검거 성공!
ㅈㅗㅈ까고 있네 진짜          ➔ 좆까고 있네 진짜          |    57.97% ➔    91.86% | 🚨 검거 성공!
야 이 씗빪아              ➔ 야 이 씌빨아            |    20.31% ➔    90.32% | 🚨 검거 성공!
시...발...새...끼        ➔ 시...발...새...끼      |    90.85% ➔    90.85% | 🚨 검거 성공!
조 까 구 있 네            ➔ 조 까 구 있 네          |    72.07% ➔    72.07% | 🚨 검거 성공!
개@새@끼                ➔ 개새끼                |    89.96% ➔    90.31% | 🚨 검거 성공!
시ㅂㅏㄹ련아               ➔ 시발련아               |    10.77% ➔    91.70% | 🚨 검거 성공!
ㅅ1ㅂㅏㄹ                ➔ ㅅ발                 |    10.77% ➔    90.36% | 🚨 검거 성공!
꺼 ㅈㅕ                 ➔ 꺼 져                |    34.92% ➔    82.01% | 🚨 검거 성공!
시베리아 귤껍데기            ➔ 시베리아 귤껍데기          |    26.39% ➔    26.39% | ❌ 여전히 실패
개색구 야                ➔ 개색구 야              |    39.22% ➔    39.22% | ❌ 여전히 실패
신발끈

In [7]:
import re
from transformers import pipeline

# ====================================================================
# [1] 영타 ➔ 한글 자모 변환 딕셔너리 (tlqkf -> 시발)
# ====================================================================
ENG_KEY_MAP = {
    'q': 'ㅂ', 'w': 'ㅈ', 'e': 'ㄷ', 'r': 'ㄱ', 't': 'ㅅ',
    'y': 'ㅛ', 'u': 'ㅕ', 'i': 'ㅑ', 'o': 'ㅐ', 'p': 'ㅔ',
    'a': 'ㅁ', 's': 'ㄴ', 'd': 'ㅇ', 'f': 'ㄹ', 'g': 'ㅎ',
    'h': 'ㅗ', 'j': 'ㅓ', 'k': 'ㅏ', 'l': 'ㅣ',
    'z': 'ㅋ', 'x': 'ㅌ', 'c': 'ㅊ', 'v': 'ㅍ', 'b': 'ㅠ',
    'n': 'ㅜ', 'm': 'ㅡ',
    'Q': 'ㅃ', 'W': 'ㅉ', 'E': 'ㄸ', 'R': 'ㄲ', 'T': 'ㅆ',
    'O': 'ㅒ', 'P': 'ㅖ'
}

def convert_eng_to_jamo(text):
    """영타로 친 단어(tlqkf 등)를 한글 자모로 변환"""
    # 순수 알파벳으로만 된 단어 변환
    words = text.split()
    converted_words = []
    for word in words:
        if re.fullmatch(r'[a-zA-Z]+', word):
            converted = "".join([ENG_KEY_MAP.get(ch, ch) for ch in word])
            converted_words.append(converted)
        else:
            converted_words.append(word)
    return " ".join(converted_words)


# ====================================================================
# [2] 한글 자모 결합기 및 정규화 엔진
# ====================================================================
CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

COMPLEX_BATCHIM_MAP = {
    'ㄼ': '',    # 씗 -> 씨
    'ㅀ': '',    # 싷 -> 시
    'ㄶ': '',    # 싾 -> 싸
    'ㄻ': 'ㄹ',  # 빪 -> 빨, 밞 -> 발
    'ㄺ': 'ㄱ',
    'ㄳ': 'ㄱ',
    'ㅄ': 'ㅂ'
}

# 발음 변형 및 사투리 매핑
SLANG_MAP = {
    "개색구": "개새끼",
    "개색기": "개새끼",
    "개새꾸": "개새끼",
    "시베리아 귤껍데기": "씨발",
    "느금마": "개새끼",
    "느검마": "개새끼" 
}

# 억울한 오탐을 막기 위한 정상 표현 화이트리스트
SAFE_PHRASES = ["수박 씨", "수박씨", "조카 신발", "조카신발", "시발점", "신발끈","애미야"]

def compose_syllable(cho, jung, jong=''):
    c_idx = CHOSUNG.index(cho)
    j_idx = JUNGSUNG.index(jung)
    t_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
    return chr(0xAC00 + (c_idx * 21 + j_idx) * 28 + t_idx)

def combine_jamo(text):
    chars = list(text)
    result = []
    i = 0
    n = len(chars)
    while i < n:
        c = chars[i]
        if c in CHOSUNG and i + 1 < n and chars[i+1] in JUNGSUNG:
            cho = c
            jung = chars[i+1]
            jong = ""
            if i + 2 < n and chars[i+2] in JONGSUNG and chars[i+2] != "":
                if i + 3 < n and chars[i+3] in JUNGSUNG:
                    jong = ""
                    i += 2
                else:
                    jong = chars[i+2]
                    i += 3
            else:
                i += 2
            result.append(compose_syllable(cho, jung, jong))
        else:
            result.append(c)
            i += 1
    return "".join(result)

def clean_bypassed_text(text):
    """최종 전처리 파이프라인"""
    # 0. 영타 욕설 변환 (tlqkf -> ㅅㅣㅂㅏㄹ)
    text = convert_eng_to_jamo(text)
    
    # 1. 특수문자/숫자 제거
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}\[\]\-.]', '', text)
    
    # 2. 쪼개진 자모 결합 (ㅅㅣ발 -> 시발)
    text = combine_jamo(text)
    
    # 3. 씗빪 같은 겹받침 단순화
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - 0xAC00
            cho = CHOSUNG[code // (21 * 28)]
            jung = JUNGSUNG[(code % (21 * 28)) // 28]
            jong = JONGSUNG[code % 28]
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_syllable(cho, jung, jong))
        else:
            normalized_chars.append(char)
    text = "".join(normalized_chars)
    
    # 4. 방언/발음 변형 치환
    for slang, clean in SLANG_MAP.items():
        text = text.replace(slang, clean)
        
    text = text.replace("씨빨", "씨발").replace("시빨", "시발")
    return text


# ====================================================================
# [3] 모델 로딩 및 다중 라벨 판정 테스트
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
print("✅ 로딩 완료!\n")

def evaluate_toxicity(text):
    """다중 라벨 중 가장 높은 혐오 점수와 라벨 반환"""
    results = classifier(text)[0]
    
    # clean을 제외한 9개 혐오/욕설 라벨 중 최고 점수 추출
    toxic_labels = [r for r in results if r['label'] != 'clean']
    top_toxic = max(toxic_labels, key=lambda x: x['score'])
    
    return top_toxic['label'], top_toxic['score'] * 100

# 🧪 아까 테스트했던 전체 문장 목록
test_sentences = [
    "야 이 ㅅㅣ1발롬아",
    "ㅅㅣㅂㅏㄹ 저리 가라",
    "ㅈㅗㅈ까고 있네 진짜",
    "야 이 씗빪아",
    "시...발...새...끼",
    "개@새@끼",
    "시ㅂㅏㄹ련아",
    "ㅅ1ㅂㅏㄹ",
    "꺼 져",
    "개색구 야",
    "시베리아 귤껍데기",
    "tlqkf",
    "느금마",
    # 오탐 방지 테스트
    "신발끈 조카신발",
    "수박 씨 발라먹어"
]

print("=" * 90)
print(f"{'원래 문장':<20} ➔ {'복원된 문장':<18} | {'최고 혐오 라벨':<10} | {'확률':<7} | {'판정'}")
print("=" * 90)

for original in test_sentences:
    # 1. 화이트리스트(오탐 방지) 체크
    is_safe = any(phrase in original for phrase in SAFE_PHRASES)
    
    # 2. 전처리 파이프라인 통과
    restored = clean_bypassed_text(original)
    
    # 3. 모델 판정 (다중 라벨 탐색)
    label, score = evaluate_toxicity(restored)
    
    # 4. 최종 판정 로직
    if is_safe and score < 90:  # 화이트리스트에 걸렸고 명백한 극단 욕설이 아니면 보호
        status = "🛡️ [오탐 방지 성공 (정상)]"
    elif score >= 50:
        status = f"🚨 검거 성공 ({label})"
    else:
        status = "❌ 감지 실패"
        
    print(f"{original:<20} ➔ {restored:<18} | {label:<10} | {score:>6.2f}% | {status}")

print("=" * 90)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

원래 문장                ➔ 복원된 문장             | 최고 혐오 라벨   | 확률      | 판정
야 이 ㅅㅣ1발롬아           ➔ 야 이 시발롬아           | 악플/욕설      |  91.76% | 🚨 검거 성공 (악플/욕설)
ㅅㅣㅂㅏㄹ 저리 가라          ➔ 시발 저리 가라           | 악플/욕설      |  90.96% | 🚨 검거 성공 (악플/욕설)
ㅈㅗㅈ까고 있네 진짜          ➔ 좆까고 있네 진짜          | 악플/욕설      |  91.86% | 🚨 검거 성공 (악플/욕설)
야 이 씗빪아              ➔ 야 이 씌빨아            | 악플/욕설      |  90.32% | 🚨 검거 성공 (악플/욕설)
시...발...새...끼        ➔ 시발새끼               | 악플/욕설      |  91.95% | 🚨 검거 성공 (악플/욕설)
개@새@끼                ➔ 개새끼                | 악플/욕설      |  90.31% | 🚨 검거 성공 (악플/욕설)
시ㅂㅏㄹ련아               ➔ 시발련아               | 악플/욕설      |  91.70% | 🚨 검거 성공 (악플/욕설)
ㅅ1ㅂㅏㄹ                ➔ ㅅ발                 | 악플/욕설      |  90.36% | 🚨 검거 성공 (악플/욕설)
꺼 져                  ➔ 꺼 져                | 악플/욕설      |  82.01% | 🚨 검거 성공 (악플/욕설)
개색구 야                ➔ 개새끼 야              | 악플/욕설      |  89.38% | 🚨 검거 성공 (악플/욕설)
시베리아 귤껍데기            ➔ 씨발                 | 악플/욕설      |  91.80% | 🚨 검거 성공 (악플/욕설)
tlqkf  

In [8]:
import re
from transformers import pipeline

# ====================================================================
# [1] 통합 전처리 및 정규화 엔진
# ====================================================================
ENG_KEY_MAP = {
    'q': 'ㅂ', 'w': 'ㅈ', 'e': 'ㄷ', 'r': 'ㄱ', 't': 'ㅅ',
    'y': 'ㅛ', 'u': 'ㅕ', 'i': 'ㅑ', 'o': 'ㅐ', 'p': 'ㅔ',
    'a': 'ㅁ', 's': 'ㄴ', 'd': 'ㅇ', 'f': 'ㄹ', 'g': 'ㅎ',
    'h': 'ㅗ', 'j': 'ㅓ', 'k': 'ㅏ', 'l': 'ㅣ',
    'z': 'ㅋ', 'x': 'ㅌ', 'c': 'ㅊ', 'v': 'ㅍ', 'b': 'ㅠ',
    'n': 'ㅜ', 'm': 'ㅡ',
    'Q': 'ㅃ', 'W': 'ㅉ', 'E': 'ㄸ', 'R': 'ㄲ', 'T': 'ㅆ',
    'O': 'ㅒ', 'P': 'ㅖ'
}

CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

COMPLEX_BATCHIM_MAP = {
    'ㄼ': '', 'ㅀ': '', 'ㄶ': '', 'ㄻ': 'ㄹ', 'ㄺ': 'ㄱ', 'ㄳ': 'ㄱ', 'ㅄ': 'ㅂ'
}

SLANG_MAP = {
    "개색구": "개새끼", "개색기": "개새끼", "개새꾸": "개새끼",
    "시베리아 귤껍데기": "씨발"
}

# ⭐️ 명백한 패드립 직권 차단 키워드 (오탐 방지)
HARD_SWEAR_WORDS = ["느금마", "느검마", "니애미", "니애비", "너네애미"]

# ⭐️ 억울한 오탐을 막기 위한 정상 표현 보호 목록
SAFE_PHRASES = ["수박 씨", "수박씨", "조카 신발", "조카신발", "시발점", "신발끈", "애미야", "어미야"]

def convert_eng_to_jamo(text):
    words = text.split()
    converted_words = []
    for word in words:
        if re.fullmatch(r'[a-zA-Z]+', word):
            converted = "".join([ENG_KEY_MAP.get(ch, ch) for ch in word])
            converted_words.append(converted)
        else:
            converted_words.append(word)
    return " ".join(converted_words)

def compose_syllable(cho, jung, jong=''):
    c_idx = CHOSUNG.index(cho)
    j_idx = JUNGSUNG.index(jung)
    t_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
    return chr(0xAC00 + (c_idx * 21 + j_idx) * 28 + t_idx)

def combine_jamo(text):
    chars = list(text)
    result = []
    i = 0
    n = len(chars)
    while i < n:
        c = chars[i]
        if c in CHOSUNG and i + 1 < n and chars[i+1] in JUNGSUNG:
            cho = c
            jung = chars[i+1]
            jong = ""
            if i + 2 < n and chars[i+2] in JONGSUNG and chars[i+2] != "":
                if i + 3 < n and chars[i+3] in JUNGSUNG:
                    jong = ""
                    i += 2
                else:
                    jong = chars[i+2]
                    i += 3
            else:
                i += 2
            result.append(compose_syllable(cho, jung, jong))
        else:
            result.append(c)
            i += 1
    return "".join(result)

def clean_bypassed_text(text):
    text = convert_eng_to_jamo(text)
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}\[\]\-.]', '', text)
    text = combine_jamo(text)
    
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - 0xAC00
            cho = CHOSUNG[code // (21 * 28)]
            jung = JUNGSUNG[(code % (21 * 28)) // 28]
            jong = JONGSUNG[code % 28]
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_syllable(cho, jung, jong))
        else:
            normalized_chars.append(char)
    text = "".join(normalized_chars)
    
    for slang, clean in SLANG_MAP.items():
        text = text.replace(slang, clean)
        
    text = text.replace("씨빨", "씨발").replace("시빨", "시발")
    return text


# ====================================================================
# [2] 모델 로딩
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
print("✅ 로딩 완료!\n")

def evaluate_toxicity(text):
    results = classifier(text)[0]
    toxic_labels = [r for r in results if r['label'] != 'clean']
    top_toxic = max(toxic_labels, key=lambda x: x['score'])
    return top_toxic['label'], top_toxic['score'] * 100


# ====================================================================
# [3] 일상 대화 10개 & 대화 속 욕설 10개 데이터셋
# ====================================================================
dataset = [
    # ---------------- [A] 일상 대화 10개 (Target: 정상 통과) ----------------
    {"text": "오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?", "is_swear": False},
    {"text": "이번 주말에 친구들이랑 오랜만에 축구하기로 했어.", "is_swear": False},
    {"text": "프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.", "is_swear": False},
    {"text": "새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.", "is_swear": False}, # '신발' 함정
    {"text": "할머니께서 애미야 밥 차려라 하고 부르시더라.", "is_swear": False},       # '애미야' 함정
    {"text": "수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.", "is_swear": False},     # '수박 씨' 함정
    {"text": "그 실패가 우리 팀의 새로운 시발점이 되었어.", "is_swear": False},        # '시발점' 함정
    {"text": "어제 본 영화 생각보다 너무 개재밌고 감동적이었어.", "is_swear": False},  # '개~' 긍정 칭찬
    {"text": "과제 다 끝내고 나니까 진짜 살 것 같네.", "is_swear": False},
    {"text": "내일 아침 9시까지 3층 회의실로 모여주세요.", "is_swear": False},

    # ---------------- [B] 대화 속 우회/변형 욕설 10개 (Target: 검거 성공) ----------------
    {"text": "게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진짜 짜증나게", "is_swear": True},
    {"text": "저 팀원 아무것도 안 하고 노는데 진짜 ㅂㅕㅇㅅㅣㄴ 같음", "is_swear": True},
    {"text": "출근길에 지하철 연착돼서 야 이 ㅅㅣ1발 지각하겠다", "is_swear": True},
    {"text": "팀장님이 또 말도 안 되는 일 시키는데 tlqkf 때려칠까", "is_swear": True},
    {"text": "어제 술 취해서 씗빪 거리던 놈 누구냐 진상이다", "is_swear": True},
    {"text": "주차를 이따위로 해놓으면 어떡하냐 개@새@끼야", "is_swear": True},
    {"text": "선 넘지 마라 진짜 시ㅂㅏㄹ련아 적당히 해", "is_swear": True},
    {"text": "걔 싸가지 없어서 선배들한테 욕먹고 꺼 져 소리 들음", "is_swear": True},
    {"text": "장난치는 것도 아니고 패드립으로 느금마 거리는 건 선 넘었지", "is_swear": True},
    {"text": "음식 배달 2시간 걸린 거 실화냐 사장 새끼 장난치나", "is_swear": True}
]


# ====================================================================
# [4] 종합 테스트 실행 및 성적표 산출
# ====================================================================
print("=" * 95)
print(f"{'#':<3} | {'테스트 문장':<38} | {'최고 혐오 라벨':<10} | {'확률':<7} | {'판정'}")
print("=" * 95)

clean_pass_count = 0     # 정상 대화인데 정상 통과한 개수 (성공)
swear_caught_count = 0   # 욕설인데 검거 성공한 개수 (성공)

for i, data in enumerate(dataset, 1):
    original = data["text"]
    is_swear = data["is_swear"]
    
    # 1. 패드립 직권 차단 체크
    is_hard_swear = any(bad in original for bad in HARD_SWEAR_WORDS)
    
    # 2. 화이트리스트 체크
    is_safe = any(phrase in original for phrase in SAFE_PHRASES)
    
    # 3. 전처리 파이프라인
    restored = clean_bypassed_text(original)
    
    # 4. 모델 판정
    label, score = evaluate_toxicity(restored)
    
    # 5. 최종 판정 로직
    if is_hard_swear:
        verdict_toxic = True
        label = "패드립/가족"
        score = 99.99
    elif is_safe and score < 90:
        verdict_toxic = False
    elif score >= 50:
        verdict_toxic = True
    else:
        verdict_toxic = False
        
    # 결과 평가
    if not is_swear:
        # 정상 대화 검증
        if not verdict_toxic:
            status = "✅ [정상 통과]"
            clean_pass_count += 1
        else:
            status = "⚠️ [오탐 발생 (오류)]"
    else:
        # 욕설 대화 검증
        if verdict_toxic:
            status = f"🚨 [검거 성공 ({label})]"
            swear_caught_count += 1
        else:
            status = "❌ [감지 실패 (뚫림)]"

    # 너무 긴 문장은 화면을 위해 자름
    display_text = (original[:35] + '..') if len(original) > 35 else original
    print(f"{i:<3} | {display_text:<38} | {label:<10} | {score:>6.2f}% | {status}")

print("=" * 95)
print("📊 [최종 종합 성적표]")
print(f"1. 정상 일상 대화 보존율 (10개 중) : {clean_pass_count}/10 ({clean_pass_count*10}%)")
print(f"2. 대화 속 우회 욕설 탐지율 (10개 중) : {swear_caught_count}/10 ({swear_caught_count*10}%)")
print(f"3. 전체 종합 정확도 (20개 중)       : {clean_pass_count + swear_caught_count}/20 ({(clean_pass_count + swear_caught_count)*5}%)")
print("=" * 95)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

#   | 테스트 문장                                 | 최고 혐오 라벨   | 확률      | 판정
1   | 오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?              | 악플/욕설      |   7.86% | ✅ [정상 통과]
2   | 이번 주말에 친구들이랑 오랜만에 축구하기로 했어.            | 악플/욕설      |   5.49% | ✅ [정상 통과]
3   | 프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.      | 악플/욕설      |   6.44% | ✅ [정상 통과]
4   | 새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.          | 악플/욕설      |   8.37% | ✅ [정상 통과]
5   | 할머니께서 애미야 밥 차려라 하고 부르시더라.              | 악플/욕설      |   8.93% | ✅ [정상 통과]
6   | 수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.            | 악플/욕설      |  12.31% | ✅ [정상 통과]
7   | 그 실패가 우리 팀의 새로운 시발점이 되었어.              | 악플/욕설      |   6.60% | ✅ [정상 통과]
8   | 어제 본 영화 생각보다 너무 개재밌고 감동적이었어.           | 악플/욕설      |   5.88% | ✅ [정상 통과]
9   | 과제 다 끝내고 나니까 진짜 살 것 같네.                | 악플/욕설      |   8.95% | ✅ [정상 통과]
10  | 내일 아침 9시까지 3층 회의실로 모여주세요.              | 악플/욕설      |   8.22% | ✅ [정상 통과]
11  | 게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진짜 짜증나게          | 악플/욕설      |  90.93% | 🚨 [검거 성공 (악플/욕설)]
12  | 저 팀원 아무것도 안 하고 노는데 진짜 ㅂ

In [9]:
import re
import time  # ⭐️ 시간 측정을 위한 라이브러리 추가
from transformers import pipeline

# ====================================================================
# [1] 통합 전처리 및 정규화 엔진
# ====================================================================
ENG_KEY_MAP = {
    'q': 'ㅂ', 'w': 'ㅈ', 'e': 'ㄷ', 'r': 'ㄱ', 't': 'ㅅ',
    'y': 'ㅛ', 'u': 'ㅕ', 'i': 'ㅑ', 'o': 'ㅐ', 'p': 'ㅔ',
    'a': 'ㅁ', 's': 'ㄴ', 'd': 'ㅇ', 'f': 'ㄹ', 'g': 'ㅎ',
    'h': 'ㅗ', 'j': 'ㅓ', 'k': 'ㅏ', 'l': 'ㅣ',
    'z': 'ㅋ', 'x': 'ㅌ', 'c': 'ㅊ', 'v': 'ㅍ', 'b': 'ㅠ',
    'n': 'ㅜ', 'm': 'ㅡ',
    'Q': 'ㅃ', 'W': 'ㅉ', 'E': 'ㄸ', 'R': 'ㄲ', 'T': 'ㅆ',
    'O': 'ㅒ', 'P': 'ㅖ'
}

CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

COMPLEX_BATCHIM_MAP = {
    'ㄼ': '', 'ㅀ': '', 'ㄶ': '', 'ㄻ': 'ㄹ', 'ㄺ': 'ㄱ', 'ㄳ': 'ㄱ', 'ㅄ': 'ㅂ'
}

SLANG_MAP = {
    "개색구": "개새끼", "개색기": "개새끼", "개새꾸": "개새끼",
    "시베리아 귤껍데기": "씨발"
}

HARD_SWEAR_WORDS = ["느금마", "느검마", "니애미", "니애비", "너네애미"]
SAFE_PHRASES = ["수박 씨", "수박씨", "조카 신발", "조카신발", "시발점", "신발끈", "애미야", "어미야"]

def convert_eng_to_jamo(text):
    words = text.split()
    converted_words = []
    for word in words:
        if re.fullmatch(r'[a-zA-Z]+', word):
            converted = "".join([ENG_KEY_MAP.get(ch, ch) for ch in word])
            converted_words.append(converted)
        else:
            converted_words.append(word)
    return " ".join(converted_words)

def compose_syllable(cho, jung, jong=''):
    c_idx = CHOSUNG.index(cho)
    j_idx = JUNGSUNG.index(jung)
    t_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
    return chr(0xAC00 + (c_idx * 21 + j_idx) * 28 + t_idx)

def combine_jamo(text):
    chars = list(text)
    result = []
    i = 0
    n = len(chars)
    while i < n:
        c = chars[i]
        if c in CHOSUNG and i + 1 < n and chars[i+1] in JUNGSUNG:
            cho = c
            jung = chars[i+1]
            jong = ""
            if i + 2 < n and chars[i+2] in JONGSUNG and chars[i+2] != "":
                if i + 3 < n and chars[i+3] in JUNGSUNG:
                    jong = ""
                    i += 2
                else:
                    jong = chars[i+2]
                    i += 3
            else:
                i += 2
            result.append(compose_syllable(cho, jung, jong))
        else:
            result.append(c)
            i += 1
    return "".join(result)

def clean_bypassed_text(text):
    text = convert_eng_to_jamo(text)
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}\[\]\-.]', '', text)
    text = combine_jamo(text)
    
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - 0xAC00
            cho = CHOSUNG[code // (21 * 28)]
            jung = JUNGSUNG[(code % (21 * 28)) // 28]
            jong = JONGSUNG[code % 28]
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_syllable(cho, jung, jong))
        else:
            normalized_chars.append(char)
    text = "".join(normalized_chars)
    
    for slang, clean in SLANG_MAP.items():
        text = text.replace(slang, clean)
        
    text = text.replace("씨빨", "씨발").replace("시빨", "시발")
    return text


# ====================================================================
# [2] 모델 로딩 (첫 웜업 포함)
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
# 초기 워밍업(첫 실행 시 지연 방지)
_ = classifier("테스트")
print("✅ 로딩 완료!\n")

def evaluate_toxicity(text):
    results = classifier(text)[0]
    toxic_labels = [r for r in results if r['label'] != 'clean']
    top_toxic = max(toxic_labels, key=lambda x: x['score'])
    return top_toxic['label'], top_toxic['score'] * 100


# ====================================================================
# [3] 일상 대화 10개 & 대화 속 욕설 10개 데이터셋
# ====================================================================
dataset = [
    {"text": "오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?", "is_swear": False},
    {"text": "이번 주말에 친구들이랑 오랜만에 축구하기로 했어.", "is_swear": False},
    {"text": "프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.", "is_swear": False},
    {"text": "새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.", "is_swear": False},
    {"text": "할머니께서 애미야 밥 차려라 하고 부르시더라.", "is_swear": False},
    {"text": "수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.", "is_swear": False},
    {"text": "그 실패가 우리 팀의 새로운 시발점이 되었어.", "is_swear": False},
    {"text": "어제 본 영화 생각보다 너무 개재밌고 감동적이었어.", "is_swear": False},
    {"text": "과제 다 끝내고 나니까 진짜 살 것 같네.", "is_swear": False},
    {"text": "내일 아침 9시까지 3층 회의실로 모여주세요.", "is_swear": False},

    {"text": "게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진짜 짜증나게", "is_swear": True},
    {"text": "저 팀원 아무것도 안 하고 노는데 진짜 ㅂㅕㅇㅅㅣㄴ 같음", "is_swear": True},
    {"text": "출근길에 지하철 연착돼서 야 이 ㅅㅣ1발 지각하겠다", "is_swear": True},
    {"text": "팀장님이 또 말도 안 되는 일 시키는데 tlqkf 때려칠까", "is_swear": True},
    {"text": "어제 술 취해서 씗빪 거리던 놈 누구냐 진상이다", "is_swear": True},
    {"text": "주차를 이따위로 해놓으면 어떡하냐 개@새@끼야", "is_swear": True},
    {"text": "선 넘지 마라 진짜 시ㅂㅏㄹ련아 적당히 해", "is_swear": True},
    {"text": "걔 싸가지 없어서 선배들한테 욕먹고 꺼 져 소리 들음", "is_swear": True},
    {"text": "장난치는 것도 아니고 패드립으로 느금마 거리는 건 선 넘었지", "is_swear": True},
    {"text": "음식 배달 2시간 걸린 거 실화냐 사장 새끼 장난치나", "is_swear": True}
]


# ====================================================================
# [4] 속도 측정 실행 및 종합 성적표
# ====================================================================
print("=" * 105)
print(f"{'#':<3} | {'테스트 문장':<36} | {'최고 라벨':<8} | {'확률':<7} | {'소요 시간':<9} | {'판정'}")
print("=" * 105)

clean_pass_count = 0
swear_caught_count = 0
total_processing_time = 0.0

for i, data in enumerate(dataset, 1):
    original = data["text"]
    is_swear = data["is_swear"]
    
    # ⭐️ 속도 측정 시작!
    start_time = time.time()
    
    # 1. 패드립 직권 차단 체크
    is_hard_swear = any(bad in original for bad in HARD_SWEAR_WORDS)
    
    # 2. 화이트리스트 체크
    is_safe = any(phrase in original for phrase in SAFE_PHRASES)
    
    # 3. 전처리 파이프라인
    restored = clean_bypassed_text(original)
    
    # 4. 모델 판정
    label, score = evaluate_toxicity(restored)
    
    # 5. 최종 판정 로직
    if is_hard_swear:
        verdict_toxic = True
        label = "패드립/가족"
        score = 99.99
    elif is_safe and score < 90:
        verdict_toxic = False
    elif score >= 50:
        verdict_toxic = True
    else:
        verdict_toxic = False
        
    # ⭐️ 속도 측정 완료 (초 단위)
    elapsed = time.time() - start_time
    total_processing_time += elapsed

    # 결과 평가
    if not is_swear:
        status = "✅ [정상 통과]" if not verdict_toxic else "⚠️ [오탐 발생]"
        if not verdict_toxic: clean_pass_count += 1
    else:
        status = f"🚨 [검거 성공 ({label})]" if verdict_toxic else "❌ [감지 실패]"
        if verdict_toxic: swear_caught_count += 1

    display_text = (original[:33] + '..') if len(original) > 33 else original
    # 소요 시간을 밀리초(ms)와 초(s) 형태로 보기 좋게 출력
    time_str = f"{elapsed*1000:>5.1f}ms"
    print(f"{i:<3} | {display_text:<36} | {label:<8} | {score:>6.2f}% | {time_str:<9} | {status}")

avg_time = (total_processing_time / len(dataset)) * 1000

print("=" * 105)
print("📊 [최종 종합 성적표 및 실시간 성능 분석]")
print(f"1. 정상 일상 대화 보존율 (10개 중) : {clean_pass_count}/10 ({clean_pass_count*10}%)")
print(f"2. 대화 속 우회 욕설 탐지율 (10개 중) : {swear_caught_count}/10 ({swear_caught_count*10}%)")
print(f"3. 전체 종합 정확도 (20개 중)       : {clean_pass_count + swear_caught_count}/20 ({(clean_pass_count + swear_caught_count)*5}%)")
print(f"⚡ 4. 평균 처리 속도 (Latency)       : 건당 {avg_time:.2f}ms (약 {avg_time/1000:.3f}초)")
print("   👉 판정: 실시간 게임/커뮤니티 채팅 서빙 가능 기준(100ms 이내)을 완벽하게 만족함!")
print("=" * 105)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

#   | 테스트 문장                               | 최고 라벨    | 확률      | 소요 시간     | 판정
1   | 오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?            | 악플/욕설    |   7.86% |  11.1ms   | ✅ [정상 통과]
2   | 이번 주말에 친구들이랑 오랜만에 축구하기로 했어.          | 악플/욕설    |   5.49% |   8.4ms   | ✅ [정상 통과]
3   | 프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.    | 악플/욕설    |   6.44% |   9.1ms   | ✅ [정상 통과]
4   | 새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.        | 악플/욕설    |   8.37% |  10.0ms   | ✅ [정상 통과]
5   | 할머니께서 애미야 밥 차려라 하고 부르시더라.            | 악플/욕설    |   8.93% |  10.8ms   | ✅ [정상 통과]
6   | 수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.          | 악플/욕설    |  12.31% |  10.6ms   | ✅ [정상 통과]
7   | 그 실패가 우리 팀의 새로운 시발점이 되었어.            | 악플/욕설    |   6.60% |  11.3ms   | ✅ [정상 통과]
8   | 어제 본 영화 생각보다 너무 개재밌고 감동적이었어.         | 악플/욕설    |   5.88% |  11.4ms   | ✅ [정상 통과]
9   | 과제 다 끝내고 나니까 진짜 살 것 같네.              | 악플/욕설    |   8.95% |  10.4ms   | ✅ [정상 통과]
10  | 내일 아침 9시까지 3층 회의실로 모여주세요.            | 악플/욕설    |   8.22% |  10.4ms   | ✅ [정상 통과]
11  | 게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진

In [10]:
import re
import time
from transformers import pipeline

# ====================================================================
# [1] 통합 전처리 및 정규화 엔진
# ====================================================================
ENG_KEY_MAP = {
    'q': 'ㅂ', 'w': 'ㅈ', 'e': 'ㄷ', 'r': 'ㄱ', 't': 'ㅅ',
    'y': 'ㅛ', 'u': 'ㅕ', 'i': 'ㅑ', 'o': 'ㅐ', 'p': 'ㅔ',
    'a': 'ㅁ', 's': 'ㄴ', 'd': 'ㅇ', 'f': 'ㄹ', 'g': 'ㅎ',
    'h': 'ㅗ', 'j': 'ㅓ', 'k': 'ㅏ', 'l': 'ㅣ',
    'z': 'ㅋ', 'x': 'ㅌ', 'c': 'ㅊ', 'v': 'ㅍ', 'b': 'ㅠ',
    'n': 'ㅜ', 'm': 'ㅡ',
    'Q': 'ㅃ', 'W': 'ㅉ', 'E': 'ㄸ', 'R': 'ㄲ', 'T': 'ㅆ',
    'O': 'ㅒ', 'P': 'ㅖ'
}

CHOSUNG = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
JUNGSUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

COMPLEX_BATCHIM_MAP = {
    'ㄳ': 'ㄱ', 
    'ㄵ': 'ㄴ', 
    'ㄶ': 'ㄴ', 
    'ㄺ': 'ㄱ', 
    'ㄻ': 'ㅁ',  # '읾' -> '임', '닮' -> '담' 처럼 'ㅁ'으로 가야 안전합니다.
    'ㄼ': 'ㅂ',  # '밞' -> '밥/바' 계열이 되도록 'ㅂ' 유지
    'ㄽ': 'ㄹ', 
    'ㄾ': 'ㄹ', 
    'ㄿ': 'ㅂ', 
    'ㅀ': 'ㄹ', 
    'ㅄ': 'ㅂ'
}

# ⭐️ 초성 및 발음 변형 자동 매핑 (신규 패턴 대거 추가!)
SLANG_MAP = {
    "개색구": "개새끼", "개색기": "개새끼", "개새꾸": "개새끼",
    "시베리아 귤껍데기": "씨발", "븅신": "병신", "븡신": "병신", "조까": "좆까", "밑칲": "미친",
    # 초성 단독 욕설 변환
    "ㅅㅂ": "시발", "ㅆㅂ": "씨발", "ㅂㅅ": "병신", "ㅈㄹ": "지랄",
    "ㅈㄲ": "좆까", "ㄲㅈ": "꺼져", "ㅁㅊ": "미친", "ㅈㄴ": "존나",
    # 숫자 결합형 변환
    "18년": "십팔년", "18새끼": "십팔새끼"
}

HARD_SWEAR_WORDS = ["느금마", "느검마", "니애미", "니애비", "너네애미", "앰창"]
SAFE_PHRASES = ["수박 씨", "수박씨", "조카 신발", "조카신발", "시발점", "신발끈", "애미야", "어미야"]

def convert_eng_to_jamo(text):
    """문장 중간에 낀 영타(wlfkf, tlqkfsus 등)를 한글 자모로 자동 변환"""
    def replace_token(match):
        eng = match.group()
        return "".join([ENG_KEY_MAP.get(ch, ch) for ch in eng])
    return re.sub(r'[a-zA-Z]+', replace_token, text)

def compose_syllable(cho, jung, jong=''):
    c_idx = CHOSUNG.index(cho)
    j_idx = JUNGSUNG.index(jung)
    t_idx = JONGSUNG.index(jong) if jong in JONGSUNG else 0
    return chr(0xAC00 + (c_idx * 21 + j_idx) * 28 + t_idx)

def combine_jamo(text):
    chars = list(text)
    result = []
    i = 0
    n = len(chars)
    while i < n:
        c = chars[i]
        if c in CHOSUNG and i + 1 < n and chars[i+1] in JUNGSUNG:
            cho = c
            jung = chars[i+1]
            jong = ""
            if i + 2 < n and chars[i+2] in JONGSUNG and chars[i+2] != "":
                if i + 3 < n and chars[i+3] in JUNGSUNG:
                    jong = ""
                    i += 2
                else:
                    jong = chars[i+2]
                    i += 3
            else:
                i += 2
            result.append(compose_syllable(cho, jung, jong))
        else:
            result.append(c)
            i += 1
    return "".join(result)

def clean_bypassed_text(text):
    # 0. 영타 변환 (wlfkf -> ㅈㅣㄹㅏㄹ)
    text = convert_eng_to_jamo(text)
    
    # 1. 기호/숫자 치환 및 제거 (지~랄 -> 지랄, 미_친 -> 미친)
    text = re.sub(r'[\d~!@#$%^&*()_+|<>?:{}\[\]\-_.]', '', text)
    
    # 2. 쪼개진 자모 결합 (ㅈㅣㄹㅏㄹ -> 지랄)
    text = combine_jamo(text)
    
    # 3. 씗빪 같은 겹받침 단순화
    normalized_chars = []
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - 0xAC00
            cho = CHOSUNG[code // (21 * 28)]
            jung = JUNGSUNG[(code % (21 * 28)) // 28]
            jong = JONGSUNG[code % 28]
            if jong in COMPLEX_BATCHIM_MAP:
                jong = COMPLEX_BATCHIM_MAP[jong]
            normalized_chars.append(compose_syllable(cho, jung, jong))
        else:
            normalized_chars.append(char)
    text = "".join(normalized_chars)
    
    # 4. 방언/초성/은어 치환
    for slang, clean in SLANG_MAP.items():
        text = text.replace(slang, clean)
        
    text = text.replace("씨빨", "씨발").replace("시빨", "시발")
    return text


# ====================================================================
# [2] 모델 로딩 (초기 웜업)
# ====================================================================
print("⏳ 모델 로딩 중...")
classifier = pipeline("text-classification", model="smilegate-ai/kor_unsmile", top_k=None)
_ = classifier("테스트")
print("✅ 로딩 완료!\n")

def evaluate_toxicity(text):
    results = classifier(text)[0]
    toxic_labels = [r for r in results if r['label'] != 'clean']
    top_toxic = max(toxic_labels, key=lambda x: x['score'])
    return top_toxic['label'], top_toxic['score'] * 100


# ====================================================================
# [3] 총 40개 데이터셋 (일상 대화 10개 + 대화 속 우회 욕설 30개)
# ====================================================================
dataset = [
    # ---------------- [A] 일상 대화 10개 (오탐 방지 대조군) ----------------
    {"text": "오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?", "is_swear": False},
    {"text": "이번 주말에 친구들이랑 오랜만에 축구하기로 했어.", "is_swear": False},
    {"text": "프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.", "is_swear": False},
    {"text": "새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.", "is_swear": False},
    {"text": "할머니께서 애미야 밥 차려라 하고 부르시더라.", "is_swear": False},
    {"text": "수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.", "is_swear": False},
    {"text": "그 실패가 우리 팀의 새로운 시발점이 되었어.", "is_swear": False},
    {"text": "어제 본 영화 생각보다 너무 개재밌고 감동적이었어.", "is_swear": False},
    {"text": "과제 다 끝내고 나니까 진짜 살 것 같네.", "is_swear": False},
    {"text": "내일 아침 9시까지 3층 회의실로 모여주세요.", "is_swear": False},

    # ---------------- [B] 기존 대화 속 우회 욕설 10개 ----------------
    {"text": "게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진짜 짜증나게", "is_swear": True},
    {"text": "저 팀원 아무것도 안 하고 노는데 진짜 ㅂㅕㅇㅅㅣㄴ 같음", "is_swear": True},
    {"text": "출근길에 지하철 연착돼서 야 이 ㅅㅣ1발 지각하겠다", "is_swear": True},
    {"text": "팀장님이 또 말도 안 되는 일 시키는데 tlqkf 때려칠까", "is_swear": True},
    {"text": "어제 술 취해서 씗빪 거리던 놈 누구냐 진상이다", "is_swear": True},
    {"text": "주차를 이따위로 해놓으면 어떡하냐 개@새@끼야", "is_swear": True},
    {"text": "선 넘지 마라 진짜 시ㅂㅏㄹ련아 적당히 해", "is_swear": True},
    {"text": "걔 싸가지 없어서 선배들한테 욕먹고 꺼 져 소리 들음", "is_swear": True},
    {"text": "장난치는 것도 아니고 패드립으로 느금마 거리는 건 선 넘었지", "is_swear": True},
    {"text": "음식 배달 2시간 걸린 거 실화냐 사장 새끼 장난치나", "is_swear": True},

    # ---------------- [C] ⭐️ 신규 추가된 우회 욕설 20개! ----------------
    {"text": "너 진짜 ㅈㅣㄹㅏㄹ하지 말고 가만히 있어라", "is_swear": True},           # 자모 분리 (지랄)
    {"text": "와 저 사람 운전하는 꼬라지 보니까 ㅁㅣㅊㅣㄴ놈이네", "is_swear": True},   # 자모 분리 (미친놈)
    {"text": "옆에서 계속 깐족거리는데 ㄲㅓㅈㅕ라 진짜", "is_swear": True},              # 자모 분리 (꺼져)
    {"text": "칭찬 고마운데 그 나이 먹고 그러고 싶냐 진짜? ㅋㅋ", "is_swear": True},  # 반어법 비꼬기
    {"text": "너 혹시 학창시절에 은따 당했냐? 말투 왜 이럼", "is_swear": True},       # 한국 사회 맥락 비하
    {"text": "진짜 지능에 문제 있는 거 인증하네", "is_swear": True},                 # 능력 비하
    {"text": "너 같은 ㅂㅕㅇㅅㅣㄴ이랑은 상종도 안 한다", "is_swear": True},             # 자모 분리 (병신)
    {"text": "일 처리 그따구로밖에 못하냐 18년아", "is_swear": True},                   # 숫자 결합 (십팔년)
    {"text": "새벽에 술 취해서 전화 거는 18새끼 진짜 극혐이다", "is_swear": True},       # 숫자 결합 (십팔새끼)
    {"text": "채팅창에서 자꾸 지~랄 거리는 애 그냥 차단 박자", "is_swear": True},        # 물결표 삽입 (지랄)
    {"text": "선 넘는 소리 작작 해라 미_친_새_끼야", "is_swear": True},                # 언더바 삽입 (미친새끼)
    {"text": "대화하는데 끼어들어서 ㅈ1ㄹ 좀 하지 마", "is_swear": True},                # 초성+숫자 (지랄)
    {"text": "아침부터 버스 놓치고 ㅆ1ㅂ 오늘 하루 망했네", "is_swear": True},           # 초성+숫자 (씨발)
    {"text": "아군 정글러 플레이 보고 ㅂㅅ 소리가 절로 나오더라", "is_swear": True},      # 순수 초성 (병신)
    {"text": "내 말에 토 달지 말고 보기 싫으니까 ㄲㅈ", "is_swear": True},               # 순수 초성 (꺼져)
    {"text": "자꾸 내 뒤에서 ㅈㄲ는 소리 좀 하지 마라", "is_swear": True},               # 순수 초성 (좆까)
    {"text": "오늘 야근 확정이라니 wlfkf 같다 진짜 퇴사각이다", "is_swear": True},       # 영타 QWERTY (지랄)
    {"text": "아직도 지가 뭘 잘못했는지 모르는 qudtls이 있네", "is_swear": True},       # 영타 QWERTY (병신)
    {"text": "자꾸 헛소리하는데 tlqkfsus아 조용히 해라", "is_swear": True},             # 영타 QWERTY (시발년)
    {"text": "아무것도 모르면서 훈수 두는 븅신들 진짜 많네", "is_swear": True},          # 발음 변형 (븅신)
    {"text": "내 앞에서 헛소리하지 말고 조까라 마이싱이다", "is_swear": True},           # 발음 변형 (조까)
    {"text": "지 부모 얼굴에 먹칠하는 ㅇㅐㅁㅊㅏㅇ 짓 좀 그만해", "is_swear": True},      # 패드립 자모 분리 (앰창)
    {"text": "읾값겖 갋짊맑셁욞 젅낡 맜없굵 밄칞 섁끾들읾갊", "is_swear": True},      # 욕 리뷰
    {"text": "이가게 가지마세요 전나 맜없고 미친 새끼들인가", "is_swear": True},      # 욕 리뷰
    {"text": "dlrkrp rkwlaktpdy wjssk aktdjqtrh alcls toRlemfdlsrk", "is_swear": True}      # 영어
]


# ====================================================================
# [4] 종합 테스트 실행 및 성적표 산출
# ====================================================================
print("=" * 105)
print(f"{'#':<3} | {'테스트 문장':<36} | {'최고 라벨':<8} | {'확률':<7} | {'소요 시간':<9} | {'판정'}")
print("=" * 105)

clean_pass_count = 0
swear_caught_count = 0
total_processing_time = 0.0

for i, data in enumerate(dataset, 1):
    original = data["text"]
    is_swear = data["is_swear"]
    
    start_time = time.time()
    
    # 1. 패드립 및 금칙어 체크
    is_hard_swear = any(bad in original for bad in HARD_SWEAR_WORDS)
    
    # 2. 화이트리스트 체크
    is_safe = any(phrase in original for phrase in SAFE_PHRASES)
    
    # 3. 전처리 파이프라인
    restored = clean_bypassed_text(original)
    
    # 4. 모델 판정
    label, score = evaluate_toxicity(restored)
    
    # 5. 최종 판정 로직
    if is_hard_swear or any(bad in restored for bad in HARD_SWEAR_WORDS):
        verdict_toxic = True
        label = "패드립/가족"
        score = 99.99
    elif is_safe and score < 90:
        verdict_toxic = False
    elif score >= 50:
        verdict_toxic = True
    else:
        verdict_toxic = False
        
    elapsed = time.time() - start_time
    total_processing_time += elapsed

    # 결과 판정
    if not is_swear:
        status = "✅ [정상 통과]" if not verdict_toxic else "⚠️ [오탐 발생]"
        if not verdict_toxic: clean_pass_count += 1
    else:
        status = f"🚨 [검거 성공 ({label})]" if verdict_toxic else "❌ [감지 실패]"
        if verdict_toxic: swear_caught_count += 1

    display_text = (original[:33] + '..') if len(original) > 33 else original
    time_str = f"{elapsed*1000:>5.1f}ms"
    print(f"{i:<3} | {display_text:<36} | {label:<8} | {score:>6.2f}% | {time_str:<9} | {status}")

avg_time = (total_processing_time / len(dataset)) * 1000
total_swears = len([d for d in dataset if d['is_swear']])
total_cleans = len([d for d in dataset if not d['is_swear']])

print("=" * 105)
print("📊 [40개 대규모 벤치마크 최종 성적표]")
print(f"1. 정상 일상 대화 보존율 ({total_cleans}개 중)   : {clean_pass_count}/{total_cleans} ({(clean_pass_count/total_cleans)*100:.1f}%)")
print(f"2. 대화 속 우회 욕설 탐지율 ({total_swears}개 중) : {swear_caught_count}/{total_swears} ({(swear_caught_count/total_swears)*100:.1f}%)")
print(f"3. 전체 종합 정확도 ({len(dataset)}개 중)         : {clean_pass_count + swear_caught_count}/{len(dataset)} ({((clean_pass_count + swear_caught_count)/len(dataset))*100:.1f}%)")
print(f"⚡ 4. 평균 처리 속도 (Latency)         : 건당 {avg_time:.2f}ms (약 {avg_time/1000:.3f}초)")
print("=" * 105)

⏳ 모델 로딩 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 로딩 완료!

#   | 테스트 문장                               | 최고 라벨    | 확률      | 소요 시간     | 판정
1   | 오늘 날씨 진짜 좋다. 점심 뭐 먹으러 갈래?            | 악플/욕설    |   7.86% |  11.3ms   | ✅ [정상 통과]
2   | 이번 주말에 친구들이랑 오랜만에 축구하기로 했어.          | 악플/욕설    |   5.49% |   9.5ms   | ✅ [정상 통과]
3   | 프로젝트 마감 기한이 얼마 안 남아서 오늘 야근할 것 같아.    | 악플/욕설    |   6.44% |  10.4ms   | ✅ [정상 통과]
4   | 새로 산 신발 신고 출근했더니 기분이 너무 상쾌하다.        | 악플/욕설    |   8.37% |  10.0ms   | ✅ [정상 통과]
5   | 할머니께서 애미야 밥 차려라 하고 부르시더라.            | 악플/욕설    |   8.93% |  10.0ms   | ✅ [정상 통과]
6   | 수박 먹을 때 수박 씨 골라내는 게 제일 귀찮아.          | 악플/욕설    |  10.37% |   9.7ms   | ✅ [정상 통과]
7   | 그 실패가 우리 팀의 새로운 시발점이 되었어.            | 악플/욕설    |   6.60% |  10.0ms   | ✅ [정상 통과]
8   | 어제 본 영화 생각보다 너무 개재밌고 감동적이었어.         | 악플/욕설    |   5.88% |  10.3ms   | ✅ [정상 통과]
9   | 과제 다 끝내고 나니까 진짜 살 것 같네.              | 악플/욕설    |   8.95% |   9.6ms   | ✅ [정상 통과]
10  | 내일 아침 9시까지 3층 회의실로 모여주세요.            | 악플/욕설    |   8.22% |   8.9ms   | ✅ [정상 통과]
11  | 게임하는데 뒤에서 자꾸 ㅈㅗㅈ까고 있네 진